# Monte Carlo Option Pricing

Monte Carlo simulation uses random numbers to generate possible future asset-price paths.

For option pricing, the basic process is:

1. Simulate possible price paths for the underlying asset.
2. Calculate the option payoff for each path.
3. Average the payoffs and discount them back to the present.

This notebook covers:
- Geometric Brownian Motion (GBM)
- European options
- Asian options


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Makes the simulation reproducible
np.random.seed(42)


## 1. Simulating Asset Prices with GBM

Under the risk-neutral model, the asset price is simulated using:

\[
S_{t+\Delta t}
=
S_t
\exp\left[
\left(r-\frac{1}{2}\sigma^2\right)\Delta t
+
\sigma\sqrt{\Delta t}Z
\right]
\]

where:
- \(S_t\) = current asset price
- \(r\) = risk-free rate
- \(\sigma\) = volatility
- \(Z\) = standard normal random variable


In [ ]:
def simulate_gbm_paths(S0, r, sigma, T, n_paths=10000, n_steps=252):
    dt = T / n_steps
    paths = np.zeros((n_paths, n_steps + 1))
    paths[:, 0] = S0

    for t in range(1, n_steps + 1):
        z = np.random.normal(size=n_paths)

        paths[:, t] = paths[:, t - 1] * np.exp(
            (r - 0.5 * sigma**2) * dt
            + sigma * np.sqrt(dt) * z
        )

    return paths


In [ ]:
# Example simulation
paths = simulate_gbm_paths(
    S0=100,
    r=0.05,
    sigma=0.20,
    T=1
)

# Plot the first 50 simulated paths
plt.figure(figsize=(10, 5))
plt.plot(paths[:50].T)
plt.title("Simulated GBM Asset Price Paths")
plt.xlabel("Trading Days")
plt.ylabel("Asset Price")
plt.grid(True)
plt.show()


## 2. European Option

A European option depends only on the asset price at maturity.

For a European call:

\[
\text{Payoff} = \max(S_T-K,0)
\]

For a European put:

\[
\text{Payoff} = \max(K-S_T,0)
\]

The Monte Carlo price is the discounted average payoff:

\[
V_0=e^{-rT}\mathbb{E}[\text{Payoff}]
\]


In [ ]:
def european_option_price(S0, K, r, sigma, T, option_type="call",
                          n_paths=10000, n_steps=252):

    paths = simulate_gbm_paths(S0, r, sigma, T, n_paths, n_steps)
    ST = paths[:, -1]

    if option_type == "call":
        payoff = np.maximum(ST - K, 0)
    else:
        payoff = np.maximum(K - ST, 0)

    return np.exp(-r * T) * payoff.mean()


call_price = european_option_price(100, 100, 0.05, 0.20, 1, "call")
put_price = european_option_price(100, 100, 0.05, 0.20, 1, "put")

print("European Call Price:", round(call_price, 3))
print("European Put Price :", round(put_price, 3))


## 3. Asian Option

An Asian option depends on the **average asset price** over the life of the option.

For a fixed-strike arithmetic-average Asian call:

\[
\text{Payoff}=\max(S_{avg}-K,0)
\]

For a fixed-strike arithmetic-average Asian put:

\[
\text{Payoff}=\max(K-S_{avg},0)
\]

where \(S_{avg}\) is the arithmetic average of the simulated asset prices.


In [ ]:
def asian_option_price(S0, K, r, sigma, T, option_type="call",
                       n_paths=10000, n_steps=252):

    paths = simulate_gbm_paths(S0, r, sigma, T, n_paths, n_steps)

    # Exclude the initial price and average the simulated observations
    average_price = paths[:, 1:].mean(axis=1)

    if option_type == "call":
        payoff = np.maximum(average_price - K, 0)
    else:
        payoff = np.maximum(K - average_price, 0)

    return np.exp(-r * T) * payoff.mean()


asian_call = asian_option_price(100, 100, 0.05, 0.20, 1, "call")
asian_put = asian_option_price(100, 100, 0.05, 0.20, 1, "put")

print("Asian Call Price:", round(asian_call, 3))
print("Asian Put Price :", round(asian_put, 3))
